<a href="https://colab.research.google.com/github/saulo-albuquerque-phys/GWgpu-jax/blob/acceptance_walk_sampler/examples/SAVEDgwgpu_jax_colab_pe_ripplegwIMRPhenomD_injection_new_priors_two_phase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GWgpu_jax — Parameter Estimation on a **synthetic injection** (Acceptance-walk Sampler, **non-uniform priors**)

This is the **injection** companion to
`gwgpu_jax_colab_pe_ripplegwIMRPhenomD_GW150914_new_priors_two_phase.ipynb`
(which runs the *real* GW150914 event). Here we **inject a GW150914-like
IMRPhenomD signal into simulated coloured Gaussian noise** and recover it —
a fast, fully deterministic inject↔recover test with a known ground truth.

Like its real-data sibling, the geometric parameters use the
**physically-motivated, non-uniform priors** from the new
`gwgpu_jax.gwgpu_jax_prior_definitions` module instead of plain uniform:

| Parameter     | Prior                 | Density        | Spec |
|---------------|-----------------------|----------------|------|
| `inclination` | isotropic orientation | `p(θ) ∝ sin θ` | `"sin"` (`SinUniform`) |
| `dec`         | isotropic sky         | `p(δ) ∝ cos δ` | `"cos"` (`CosUniform`) |
| `distance`    | uniform in volume     | `p(d) ∝ d²`    | `"volumetric"` (`Volumetric`) |

These are passed via the opt-in `priors=` keyword of
`gwgpu_jax.GWgpu_jaxAcceptanceWalkSampler` (its default is the unchanged uniform
behaviour). Everything is pure JAX / `jit`-traceable, so the sampler
hot path keeps its full speed — the non-uniform density is supplied as the
sampler's `logprior_fn`, and the initial live points are drawn from it by
inverse-CDF (no rejection).

The **acceptance-walk sampler** (Prathaban et al. 2025) replaces each
lowest-likelihood live point with a fresh one drawn by the `bilby`/`dynesty`
constrained random walk on the unit hypercube (`n_target` accepted steps per new
point, `num_delete` points replaced per iteration, batched on GPU). It is the
LIGO/Virgo-standard nested-sampling kernel, GW-validated within `blackjax-ns`;
the likelihood, waveform, and priors are unchanged.

**Runtime → Change runtime type → T4 GPU** before running. Total wall-clock
is a few minutes on a T4.

## 1. Install GWgpu_jax

GWgpu_jax currently lives in a **private** GitHub repo, so `pip` needs a Personal Access Token (PAT) to clone it. Two ways to provide one:

- **(Recommended) Colab Secrets** — click the key icon 🔑 in the left sidebar → *Add new secret* → name it `GH_TOKEN`, paste your PAT, and toggle *Notebook access* on. The next cell will pick it up automatically and you never see a prompt.
- **One-off prompt** — if no Colab secret is set, the cell falls back to `getpass.getpass()` so you can paste the token without it appearing in the notebook output.

Generate a PAT at <https://github.com/settings/tokens?type=beta>. A **fine-grained** token scoped to `saulo-albuquerque-phys/GWgpu-jax` with **Contents: read-only** is enough.

In [ ]:
import importlib.util, os

RUNNING_ON_COLAB = importlib.util.find_spec("google.colab") is not None
ALREADY_INSTALLED = importlib.util.find_spec("gwgpu_jax") is not None

if RUNNING_ON_COLAB and not ALREADY_INSTALLED:
    import getpass
    OWNER, REPO, BRANCH = "saulo-albuquerque-phys", "GWgpu-jax", "acceptance_walk_sampler"

    # Pin JAX to the 0.4.x line Colab's CUDA plugin still supports.
    !pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
    !pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

    # GitHub PAT via Colab Secrets (key icon in the sidebar) or getpass.
    # NEVER hard-code a token here — it leaks the moment the notebook is shared.
    GH_TOKEN = None
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not GH_TOKEN:
        GH_TOKEN = getpass.getpass(f"GitHub PAT (for {OWNER}/{REPO}): ")
    os.environ["GH_TOKEN"] = (GH_TOKEN or "").strip()
    if not os.environ["GH_TOKEN"]:
        raise RuntimeError("Empty GitHub PAT — set the GH_TOKEN Colab secret or paste a fine-grained PAT.")

    !pip install -q "gwgpu_jax[data] @ git+https://$GH_TOKEN@github.com/{OWNER}/{REPO}.git@{BRANCH}"
    !pip install -q corner

    del os.environ["GH_TOKEN"]
    del GH_TOKEN
    print("Installed. If JAX was re-pinned, use Runtime -> Restart session, then re-run from here "
          "(including the kernel-fetch cell below).")
else:
    print("GWgpu_jax already importable - skipping install.")


In [ ]:
# ── Fetch the external acceptance-walk kernel (Prathaban et al. 2025) ────────
# Cheap and idempotent: safe to re-run on its own (e.g. after a runtime restart)
# WITHOUT reinstalling gwgpu_jax. The kernel is a separate, separately-licensed
# repo (mrosep/blackjax_ns_gw) and is NOT bundled with gwgpu_jax.
import importlib.util, os, sys, subprocess

if importlib.util.find_spec("custom_kernels") is None:
    KDIR = "/content/blackjax_ns_gw" if os.path.isdir("/content") else "external/blackjax_ns_gw"
    if not os.path.isdir(f"{KDIR}/src/custom_kernels"):
        subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", "--filter=blob:none",
                        "https://github.com/mrosep/blackjax_ns_gw.git", KDIR], check=True)
        subprocess.run(["git", "-C", KDIR, "sparse-checkout", "set", "src/custom_kernels"], check=True)
        subprocess.run(["git", "-C", KDIR, "checkout"], check=True)
    sys.path.insert(0, f"{KDIR}/src")
    os.environ["GWJAX_ACCEPTANCE_WALK_SRC"] = f"{KDIR}/src"

import custom_kernels
print("acceptance-walk kernel ready:", custom_kernels.__file__)


In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Enable double precision — important for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwgpu_jax
print("gwgpu_jax version :", gwgpu_jax.__version__)
print("JAX devices   :", jax.devices())

## 2. Build a detector network

We use a 4-second segment at 2048 Hz, analysing the 20–512 Hz band. The `TimeFrequencyGrid` is shared across all detectors in the network.

In [ ]:
DURATION       = 4.0
SAMPLING_RATE  = 2048.0
F_MIN, F_MAX   = 20.0, 512.0

grid = gwgpu_jax.TimeFrequencyGrid(
    duration=DURATION, sampling_rate=SAMPLING_RATE,
    f_min=F_MIN, f_max=F_MAX,
)
network = gwgpu_jax.Network.from_names(["H1", "L1"], grid)
print(grid)
print(network)

## 3. Synthetic injection (fast, deterministic)

Generate coloured Gaussian noise at the aLIGO design PSD, then inject a
GW150914-like IMRPhenomD signal with the known `TRUE_PARAMS` below. The
network optimal SNR ends up around 19. Because the truth is known, the
summary and corner plot overlay it so you can check the posterior recovers
the injected values.

In [ ]:
TRUE_PARAMS = dict(
    m1=35.0, m2=30.0, chi_1=0.0, chi_2=0.0,
    distance=410.0, inclination=0.4,
    tc=0.0, phi_c=0.0,
    ra=1.375, dec=-1.21, psi=0.0,
)

network.generate_noise(seed=0)
waveform_fn = gwgpu_jax.build_ripplegw_waveform_fn("IMRPhenomD", f_ref=20.0)

hp, hc = waveform_fn(TRUE_PARAMS, grid.frequency_domain_array)
# Project with the SAME tc+dt_ifo convention as the sampler's likelihood, so the
# injected merger sits at TRUE_PARAMS["tc"]. network.project_waveform applies ONLY
# the geometric delay dt_ifo (not tc); it would place the signal at tc=0 — fine
# while tc=0, but it silently breaks for any nonzero tc.
_freqs = grid.frequency_domain_array
_gmst  = float(network.gmst)
h_dict = {}
for ifo in network.interferometers:
    Fp, Fc = ifo.antenna_pattern(TRUE_PARAMS["ra"], TRUE_PARAMS["dec"], TRUE_PARAMS["psi"], _gmst)
    dt_ifo = ifo.time_delay_from_geocenter(TRUE_PARAMS["ra"], TRUE_PARAMS["dec"], _gmst)
    h_dict[ifo.name] = gwgpu_jax.waveform_projection_fd(
        hp, hc, Fp, Fc, _freqs, TRUE_PARAMS["tc"] + dt_ifo)
for name, snr in network.optimal_snr(h_dict).items():
    print(f"  injected {name} SNR = {float(snr):.1f}")
print(f"  network SNR = {float(network.network_optimal_snr(h_dict)):.1f}")

network.inject_signal(h_dict, domain="fd")

## 4. Set up the acceptance-walk nested sampler (with non-uniform priors)

An 11-dimensional prior: component masses, **aligned spins** (`chi_1`,
`chi_2`), distance, inclination, and the sky/time parameters.

The change from the uniform notebook is the new **`priors=`** argument. Each
entry of `PARAM_BOUNDS` still sets the parameter's support `(a, b)`; `priors`
then overrides the *shape* of the density on that interval for the geometric
parameters:

- `"inclination": "sin"` → `p(θ) ∝ sin θ` (isotropic binary orientation),
- `"dec": "cos"` → `p(δ) ∝ cos δ` (isotropic sky declination),
- `"distance": "volumetric"` → `p(d) ∝ d²` (sources uniform in volume).

Every other parameter (masses, spins, `ra`, `psi`, `phi_c`, `tc`) keeps its
default **uniform** prior — any name absent from `priors` is left uniform.

In [ ]:
# tc convention (synthetic injection).
# The injection (cell 7) applies tc explicitly (tc + dt_ifo per IFO), so the
# merger sits at t = TRUE_PARAMS["tc"]. With tc=0 here the prior is a small
# ±50 ms window around 0; a nonzero TRUE_PARAMS["tc"] is now recovered too.
TC_CENTER, TC_HALFWIDTH = 0.0, 0.05

# Sample (m1, m2) and the aligned spins (chi_1, chi_2). IMRPhenomD is
# symmetric under the label swap (m1, chi_1) ↔ (m2, chi_2), so the posterior
# has two equally-likely modes. We collapse them in cell 15 with a
# heavier-first relabelling that carries each spin with its mass — no
# information is lost, and the medians snap to (m1 = heavier, m2 = lighter).
PARAM_BOUNDS = {
    "m1":          (10.0, 80.0),
    "m2":          (10.0, 80.0),
    "chi_1":       (-0.9,  0.9),
    "chi_2":       (-0.9,  0.9),
    "distance":    (50.0, 1500.0),
    "inclination": (0.0,  float(jnp.pi)),
    "ra":          (0.0,  2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0,  float(jnp.pi)),
    "phi_c":       (0.0,  2.0 * float(jnp.pi)),
    "tc":          (TC_CENTER - TC_HALFWIDTH, TC_CENTER + TC_HALFWIDTH),
}
FIXED_PARAMS = {}

# ── NEW: non-uniform priors on the geometric parameters ─────────────────
# String aliases resolved by gwgpu_jax.gwgpu_jax_prior_definitions over the
# matching PARAM_BOUNDS support. Equivalent explicit form:
#   from gwgpu_jax import SinUniform, CosUniform, Volumetric
#   PRIORS = {
#       "inclination": SinUniform(0.0, float(jnp.pi)),
#       "dec":         CosUniform(-float(jnp.pi)/2, float(jnp.pi)/2),
#       "distance":    Volumetric(50.0, 1500.0),
#   }
PRIORS = {
    "inclination": "sin",          # p(θ) ∝ sin θ   (isotropic orientation)
    "dec":         "cos",          # p(δ) ∝ cos δ   (isotropic sky)
    "distance":    "volumetric",   # p(d) ∝ d²      (uniform in volume)
}

sampler = gwgpu_jax.GWgpu_jaxAcceptanceWalkSampler(
    network       = network,
    waveform_fn   = waveform_fn,
    param_bounds  = PARAM_BOUNDS,
    fixed_params  = FIXED_PARAMS,
    priors        = PRIORS,                        # same non-uniform priors as NSS
    periodic_params = ("phi_c", "ra"),             # wrap these angles on the unit cube
    gmst          = None,                          # auto-uses network.gmst (= 0.0 for synthetic injections)
)
print(f"sampling dimension: {len(sampler.param_bounds)}")
print("priors:", {k: type(v).__name__ for k, v in sampler._prior_specs.items()})
print(f"tc prior centred at {TC_CENTER:.3f} s (synthetic-injection convention)")

## 5. Run the acceptance-walk sampler

Tuning notes:

- `phase1_num_delete` should sit between `num_live // 100` (very safe, modest speedup) and `num_live // 20` (faster, slight bias from the larger batch). Default below is `num_live // 20 = 25`.
- `phase1_delta_logz_threshold` controls when to switch to phase 2. `-1.0` means *the live points still hold ≈ 37 % of total `Z`*; `-2.0` means ≈ 14 %; `0.0` switches as soon as the live evidence starts shrinking.
- `phase2_num_delete = 1` is the classical Skilling choice (unbiased).
- `log_dlogz_target = -3.0` is the final convergence tolerance for phase 2.

**Compilation warning.** Each phase compiles a separate XLA program (the `num_delete` is baked in), so you will see two "JIT-compiling … kernel" messages — typically 30–90 s each on a CPU runtime, much faster on a T4 GPU. Steady-state iterations after that are *fast*.

In [ ]:
import time

NUM_LIVE   = 2000
N_TARGET   = 60      # accepted MCMC walk steps per new point (bilby nact-style); raise for tighter posteriors
MAX_MCMC   = 5000    # hard cap on proposals per new point
NUM_DELETE = 10      # live points replaced per NS step (batched on GPU; 1 = lowest bias)

t0 = time.perf_counter()
result = sampler.run(
    rng_key               = jax.random.PRNGKey(0),
    num_live              = NUM_LIVE,
    n_target              = N_TARGET,
    max_mcmc              = MAX_MCMC,
    num_delete            = NUM_DELETE,
    max_iterations        = 20000,
    log_dlogz_target      = -3.0,
    num_posterior_samples = 2000,
    verbose               = True,
)
elapsed = time.perf_counter() - t0

print(f"\ntotal wall-clock   = {{elapsed:.1f}} s")
print(f"  iterations       = {{result.n_iterations}}")
print(f"  log Z            = {{result.logZ:+.3f}} ± {{result.logZ_err:.3f}}")
print(f"  ESS              = {{result.ess:.1f}}")


## 6. Posterior summary

If you see ⚠ collapsed columns / very low ESS below, the NS budget was too
small for the dimensionality. Bump `NUM_LIVE` to 800–1000 or `PHASE2_INNER_STEPS`
to 250 and re-run cells 11 → 13.

In [ ]:
# Heavier-first relabelling.
# IMRPhenomD is symmetric under (m1, chi_1) ↔ (m2, chi_2), so the raw
# posterior has two equally-likely label-swap modes. A single boolean swap
# maps every sample to the heavier-first convention, carrying each spin with
# its mass — unambiguous, information-preserving, vectorised over samples.
_m1, _m2 = result.posterior_samples["m1"], result.posterior_samples["m2"]
_c1, _c2 = result.posterior_samples["chi_1"], result.posterior_samples["chi_2"]
swap = _m2 > _m1                       # samples labelled lighter-first
# result.posterior_samples is a plain dict — mutate in place (the result is a
# NamedTuple, so the field itself can't be rebound). Carry each spin with its
# mass so chi_1 always refers to the heavier BH.
result.posterior_samples["m1"]    = jnp.where(swap, _m2, _m1)
result.posterior_samples["m2"]    = jnp.where(swap, _m1, _m2)
result.posterior_samples["chi_1"] = jnp.where(swap, _c2, _c1)
result.posterior_samples["chi_2"] = jnp.where(swap, _c1, _c2)

print(f"  {'param':12s}  {'median':>10s}  {'-1σ':>8s}  {'+1σ':>8s}  truth")
degenerate = []
for name in sampler.param_bounds:
    s = jnp.asarray(result.posterior_samples[name])
    lo, mid, hi = jnp.percentile(s, jnp.array([16.0, 50.0, 84.0]))
    if (hi - lo) < 1e-12 * jnp.maximum(jnp.abs(mid), 1.0):
        degenerate.append(name)
    truth_str = f"{TRUE_PARAMS[name]:+.3f}" if TRUE_PARAMS is not None else "—"
    print(f"  {name:12s}  {float(mid):+10.3f}  {float(mid-lo):8.3f}  {float(hi-mid):8.3f}  {truth_str}")

if degenerate:
    print(
        f"\n⚠  posterior columns collapsed (no spread): {degenerate}\n"
        f"   ESS={result.ess:.1f}  →  NS likely didn't converge.\n"
        f"   Bump NUM_LIVE to 800 and PHASE2_INNER_STEPS to 250, then re-run cell 13."
    )

## 7. Corner plot

The plot range is forced to the prior bounds via `range=`, so the plot always
renders even when the posterior is degenerate — but in that case the
histograms will look like spikes (use the summary table above to diagnose).

In [ ]:
import corner

names  = list(sampler.param_bounds.keys())
data   = np.column_stack([np.asarray(result.posterior_samples[n]) for n in names])
truths = [TRUE_PARAMS[n] for n in names] if TRUE_PARAMS is not None else None
ranges = [sampler.param_bounds[n] for n in names]

fig = corner.corner(
    data, labels=names, truths=truths,
    range=ranges,
    quantiles=[0.16, 0.5, 0.84], show_titles=True,
    title_kwargs={"fontsize": 10},
)
fig.set_size_inches(11, 11)
plt.show()

## What next?

- **Tune `phase1_num_delete`**: increase toward `num_live // 10` for more wall-clock speedup; decrease toward `num_live // 100` to minimise bias from the batched-delete approximation.
- **Tune `phase1_delta_logz_threshold`**: more negative (e.g. `-2.0` or `-3.0`) means *spend more time in the cheap regime*; closer to `0` means *transition early and rely on phase 2 for the bulk*.
- **Use your own waveform**: wrap any pure-JAX `hp, hc = wf(freqs, **params)` in `gwgpu_jax.CustomWaveform` and pass it as `waveform_fn=` (no need for ripplegw).
- **Add more detectors**: include `"V1"`, `"K1"`, or even `"ET"`/`"CE"` in the `Network.from_names` list.
- **Fix the spins** (faster, 9-D): move `chi_1`, `chi_2` back into `FIXED_PARAMS = {"chi_1": 0.0, "chi_2": 0.0}` for an aligned-spin-zero run.
- **Run locally**: same workflow as a CLI script —
  ```bash
  python examples/run_pe_local.py --two-phase \
      --num-live 500 --num-inner-steps 40 \
      --phase1-num-delete 25 --phase1-delta-logz-threshold -1.0 \
      --phase2-num-delete 1 --max-iterations 5000 --log-dlogz-target -3.0
  ```
- **Single-phase variant**: the canonical reference is
  [`examples/gwgpu_jax_colab_pe.ipynb`](https://github.com/saulo-albuquerque-phys/GWgpu-jax/blob/main/examples/gwgpu_jax_colab_pe.ipynb).